**Tansform Races Data**
1. Read bronze Races table
2. Keep only the columns required for analytics.
3. standardise column names using snake_case.
4. Rename columns to make them more meaningful.
6. Remobe duplicate records.
7. Transform values of columns race_name and locality to Title case
8. write the trandformed data to silver circuits table

In [0]:
%run ../00.Common/01.environment-config


In [0]:
bronze_table = f"{catalog_nameone}.{bronze_schema}.races"
silver_table = f"{catalog_nameone}.{silver_schema}.races"

In [0]:
races_df = spark.read.table(bronze_table)

display(races_df)

In [0]:
from pyspark.sql.functions import col

races_selected_df = races_df.select(
col("season"),
col("round"),
col("raceName"),
col("date"),
col("circuitId"),
col("ingestion_timestamp"),
col("source_file")
)

In [0]:
races_renamed_df = (
  races_selected_df
    .withColumnsRenamed({
      "circuitId": "circuit_id",
      "raceName": "race_name",
      "date": "race_date", 
    })
)

In [0]:
races_distinct_df = races_renamed_df.dropDuplicates(["season", "round"])

In [0]:
from pyspark.sql import functions as F
races_final_df = (
    races_distinct_df
        .withColumn('race_name', F.initcap(F.col("race_name")))
    
)

In [0]:
races_final_df.write.format("delta").mode("overwrite").saveAsTable(silver_table)

In [0]:
display(spark.table(silver_table))